<a href="https://colab.research.google.com/github/udayRage/cuPAMI/blob/main/notebooks/periodicFrequentPattern/cuPFEclat.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding periodic frequent pattern using cuPFEclat

This tutorial shows how to prepare a dataset, run cuPFEclat, inspect the generated patterns, and save the results.

[Algorithm documentation](https://github.com/udayRage/cuPAMI/blob/main/docs/algorithms/cuPFEclat.md)

# Prerequisites

1. Installing cuPAMI and CuPy

Select a GPU runtime in Colab. The following cell installs missing packages. The CuPy package shown is for CUDA 12.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cupami") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/udayRage/cuPAMI.git"])
if importlib.util.find_spec("cupy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cupy-cuda12x"])

2. Checking the GPU

In [2]:
import cupy as cp

print("CUDA devices:", cp.cuda.runtime.getDeviceCount())


CUDA devices: 2


3. Preparing a sample dataset

Use TemporalDatabase with a fixed seed so the input can be reproduced.

In [3]:
from cupami import TemporalDatabase, cuPFEclat

generator = TemporalDatabase(transactions=100, items=8, avg_length=3, seed=42)
data = generator.create()
print(data[:5])


[[1, 2, 5, 6, 7], [2, 1, 2, 4, 5], [3, 1], [4, 2, 4, 7], [5, 1, 2, 7]]


## Finding patterns using cuPFEclat

### Step 1: Choosing the input parameters

`minSup=0.1` means 10% of transactions. `maxPer=30` allows gaps of at most 30 timestamp units, including the start and end gaps.

### Step 2: Executing cuPFEclat

In [4]:
miner = cuPFEclat(data, minSup=0.1, maxPer=30)
miner.mine()


### Step 3: Investigating the generated patterns

Display the number of patterns and the first ten results.

In [5]:
from itertools import islice

patterns = miner.getPatterns()
print("Patterns:", len(patterns))
for pattern, support in islice(patterns.items(), 10):
    print(pattern, support)


Patterns: 33
1 (44, 9)
2 (45, 9)
3 (34, 9)
4 (29, 13)
5 (39, 8)
6 (48, 8)
7 (36, 9)
8 (37, 11)
1	2 (22, 18)
1	3 (18, 13)


### Step 4: Saving patterns and measuring runtime

Save the patterns to `patterns.txt`. Loading, mining, and total runtime are reported in seconds.

In [6]:
print("Loading:", miner.getLoadingTime())
print("Mining:", miner.getMiningTime())
print("Total:", miner.getTotalRuntime())
miner.save("patterns.txt")


Loading: 0.00040958099998533726
Mining: 0.17367998301051557
Total: 0.1740895640105009
